# Simple Nonlinear Node

This tutorial shows how to extend RxInfer with a factor node of your own and the message update rules that make it work. The node is a nonlinear function, and its rules approximate the function's messages with samples.

Writing rules by hand is worth it when you know something the general machinery does not: a closed form, a faster computation, or an approximation you want to control. Here the approximation is sampling, which represents beliefs of any shape, several modes included. RxInfer can also infer a node that has no rules at all, by approximating every belief with a Gaussian; the closing section points to those examples.

In [ ]:
using RxInfer, Random, StableRNGs

## The node

The node is deterministic, $w = f(\theta)$, with two interfaces: the output `out` and the input `in`. A node is named by a type; this one needs no fields, since the function is not part of the node but of the algorithm the rules run under.

The algorithm, `NonlinearApproximation`, carries what the rules need: the function and the number of samples. Its type extends `DefaultAlgorithmExtension`, so that RxInfer's own rules still apply to every other node.

In [ ]:
struct NonlinearNode end # the node's name

# The algorithm the node's rules run under, and their parameters; the rules draw their
# samples from the random number generator the engine gives them
struct NonlinearApproximation{F} <: DefaultAlgorithmExtension
    fn       :: F   # the nonlinear function, one number in, one number out
    nsamples :: Int # the number of samples of an approximation
end

In [ ]:
@define_factor_node(node = NonlinearNode, type = Deterministic, interfaces = [:out, :in])

## The rules

A deterministic node needs a rule for each direction, under belief propagation:

- **Towards `out`, from the message on `in`.** The message on `out` is the distribution of $f(\theta)$ when $\theta$ follows the message on `in`. We draw samples of $\theta$, push them through $f$, and return them as a `SampleList`, a distribution represented by weighted samples. The rule asks the engine for its random number generator with `ctx = (:rng,)`, so that a run is reproducible.
- **Towards `in`, from the message on `out`.** The message on `in` is the message on `out` evaluated at $f(\theta)$, a function of $\theta$ that is not a known distribution. We return it as a log-density, `ContinuousUnivariateLogPdf`. RxInfer multiplies it with the other messages on $\theta$, and the form constraint below turns that product into samples.

Each rule names its algorithm, and receives it as `algo`.

In [ ]:
# Rule for outbound message on `out` edge given inbound message on `in` edge
@define_message_update_rule(
    node = NonlinearNode, target = :out, algorithm = NonlinearApproximation,
    args = (m[:in]::NormalMeanVariance,),
    ctx = (:rng,),
    body = (algo, ctx, args) -> begin
        samples = rand(ctx.rng, args.m[:in], algo.nsamples)
        return SampleList(map(algo.fn, samples))
    end,
)

In [ ]:
# Rule for outbound message on `in` edge given inbound message on `out` edge
@define_message_update_rule(
    node = NonlinearNode, target = :in, algorithm = NonlinearApproximation,
    args = (m[:out]::Gamma,),
    body = (algo, args) -> ContinuousUnivariateLogPdf((x) -> logpdf(args.m[:out], algo.fn(x))),
)

The rule towards `in` takes a `Gamma` message on `out`: in the model below, `out` is the precision of a Gaussian, whose rules send a Gamma message towards it.

## The model

We estimate the precision of Gaussian observations, but through a latent variable $\theta$ on the whole real line: the precision is $w = f(\theta)$, with a function that makes any $\theta$ a nonnegative precision.

$$\begin{aligned}
p(\theta) &= \mathcal{N}(\theta \mid \mu_{\theta}, \sigma_{\theta}),\\
p(m) &= \mathcal{N}(m \mid \mu_{m}, \sigma_{m}),\\
w &= f(\theta),\\
p(y_i \mid m, w) &= \mathcal{N}(y_i \mid m, w^{-1}).
\end{aligned}$$

The function is $f(\theta) = |e^{\theta} \sin\theta|$. It is not one-to-one: several values of $\theta$ give the same precision. The data determine $w$, and the posterior of $\theta$ can then have several modes, one where $f$ reaches each value.

In [ ]:
@model function nonlinear_estimation(y, θ_μ, m_μ, θ_σ, m_σ)

    # the priors of the two latent variables
    θ ~ Normal(mean = θ_μ, variance = θ_σ)
    m ~ Normal(mean = m_μ, variance = m_σ)

    # the nonlinear node
    w ~ NonlinearNode(θ)

    # the observations are Gaussian with mean m and precision w
    for i in eachindex(y)
        y[i] ~ Normal(mean = m, precision = w)
    end

end

The constraints say how the posterior factorises and what form its factors take:

- `q(θ, w, m) = q(θ)q(m)q(w)` is the mean-field factorisation, under which RxInfer runs variational message passing between $m$ and $w$;
- `q(θ) :: SampleListFormConstraint(nsamples, LeftProposal())` represents $\theta$'s posterior by samples, drawn from the prior, the left factor of the product, and weighted by the log-density message, the right one;
- `q(w) :: SampleListFormConstraint(nsamples, RightProposal())` represents $w$'s posterior by samples drawn from the `SampleList` message.

In [ ]:
@constraints function nonlinear_constraints(nsamples)
    q(θ) :: SampleListFormConstraint(nsamples, LeftProposal())
    q(w) :: SampleListFormConstraint(nsamples, RightProposal())

    q(θ, w, m) = q(θ)q(m)q(w)
end

The algorithm block attaches `NonlinearApproximation` to every `NonlinearNode` between $\theta$ and $w$, with the function and the number of samples:

In [ ]:
@algorithm function nonlinear_algorithm(fn, nsamples)
    NonlinearNode(θ, w) -> NonlinearApproximation(fn, nsamples)
end

Variational message passing between $m$ and $w$ needs a marginal of each to start from:

In [ ]:
@initialization function nonlinear_initialization()
    q(m) = vague(NormalMeanPrecision)
    q(w) = vague(Gamma)
end

## Data

In [ ]:
nonlinear_fn(x) = abs(exp(x) * sin(x))

In [ ]:
seed = 123
rng  = StableRNG(seed)

niters   = 15 # Number of VMP iterations
nsamples = 5_000 # Number of samples in approximation

n = 500 # Number of IID samples
μ = -10.0
θ = -1.0
w = nonlinear_fn(θ)

data = rand(rng, NormalMeanPrecision(μ, w), n);

## Inference

In [ ]:
result = infer(
    model = nonlinear_estimation(θ_μ = 0.0, m_μ = 0.0, θ_σ = 100.0, m_σ = 1.0),
    algorithm = nonlinear_algorithm(nonlinear_fn, nsamples),
    options = (context = (rng = StableRNG(123),),), # the generator the rules draw from
    constraints = nonlinear_constraints(nsamples),
    data = (y = data, ),
    initialization = nonlinear_initialization(),
    returnvars = (θ = KeepLast(), m = KeepLast()),
    iterations = niters,
    showprogress = true
)

The posterior of $\theta$ is a `SampleList`:

In [ ]:
θposterior = result.posteriors[:θ]

## Checking the result

The data determine the precision $w = f(\theta)$ and the mean $m$, not $\theta$ itself, so these are what we check against the true values. The expectation of a function under a `SampleList` is its weighted average over the samples, `mean(f, q)`.

In [ ]:
mposterior = result.posteriors[:m]
w_estimated = mean(nonlinear_fn, θposterior)
@assert abs(w_estimated - w) < 0.05 # hide
@assert abs(mean(mposterior) - μ) < 0.2 # hide
println("precision: estimated ", round(w_estimated; digits = 3), ", true ", round(w; digits = 3))
println("mean:      estimated ", round(mean(mposterior); digits = 3), " ± ", round(std(mposterior); digits = 3), ", true ", μ, ", the data's average ", round(sum(data) / n; digits = 3))

The histogram of the samples of $\theta$, above the function, shows that they gather where $f(\theta)$ is close to the true precision: most between $-1.25$ and $-0.5$, around the true $\theta = -1$, and a few near $0.25$, where $f$ takes the same value. The posterior has two modes, and the samples represent both.

In [ ]:
using Plots

samples, weights = collect(RxInfer.BayesBase.get_samples(θposterior)), collect(RxInfer.BayesBase.get_weights(θposterior))
@assert sum(weights[-1.5 .< samples .< -0.25]) > 0.8 && sum(weights[0.0 .< samples .< 0.5]) > 0.02 # hide
p1 = histogram(samples; weights = weights, normalize = :pdf, bins = range(-3, 3, length = 121), xlims = (-3, 3), label = "q(θ), samples", fillalpha = 0.4, color = :gray, title = "Posterior for θ")
vline!(p1, [θ]; color = :black, label = "true θ")
p2 = plot(range(-3, 3, length = 400), nonlinear_fn; xlims = (-3, 3), ylims = (0, 1), color = :black, label = "f(θ)", xlabel = "θ")
hline!(p2, [w]; color = :red, linestyle = :dash, label = "true precision w")
plot(p1, p2; layout = (2, 1), size = (800, 550))

## Without writing rules

The same model can be written without a node or rules of its own: the likelihood as a node defined only by its `logpdf`, $\log\mathcal{N}(y_i \mid m, f(\theta)^{-1})$, and `infer(...; fallback = LaplaceApproximation())` or `fallback = CVI()`, which approximate every belief with a Gaussian, or a Gaussian in transformed coordinates. A Gaussian has one mode, though, and the posterior of $\theta$ here can have several; the samples of the rules above represent them all, which is a reason to write rules by hand for this node. The *Universal Inference* examples show the fallback at work:

- [Laplace Approximation: Logistic Regression](https://examples.rxinfer.com/categories/universal_inference/laplace_logistic_regression/) introduces it on a model with an exact answer;
- [Laplace and CVI: A Hierarchical Model](https://examples.rxinfer.com/categories/universal_inference/laplace_hierarchical_model/) uses it where every node has rules but one posterior has no closed form;
- [Laplace, CVI and EP: A Pendulum Filter and Smoother](https://examples.rxinfer.com/categories/universal_inference/laplace_pendulum/) computes nonlinear functions in a model with no approximation method chosen per node.